# Publication figures

Centralized notebook for all paper / presentation figures. Each figure is a standalone cell with no shared state assumptions. All figures save as PDF (vector) to `derivatives/figures/`. Add `also_png=True` to `save_fig()` for a PNG copy.

**Styling** is set once at the top by `setup_style()` from `figure_style.py`. Edit that module to change colors, fonts, or sizes globally.

**Figure index**:

| # | Figure | Source data |
|---|--------|-------------|
| 0 | Atlas visualization | nilearn atlas |
| 1 | Trial structure schematic | (no data) |
| 2 | Per-subject motion / QC summary | `derivatives/parcels/<atlas>/*_provenance.json` |
| 3 | Probe-dyscontrol within-subject correlations | `block_features.csv` |
| 4 | LOSO decoding result + permutation null | `decoding/loso_per_subject.csv` + `loso_permutation_null.csv` |
| 5 | Parcel coefficient interpretation | `decoding/parcel_coefficients.csv` |
| 6 | Decoder × probe correlations | `decoding/probe_validation.csv` |

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

sys.path.insert(0, str(Path.cwd()))

from config import (
    BASE, DERIVATIVES_DIR, PARCELS_DIR, BLOCK_FEATURES_CSV,
    ATLAS_NAME, ATLAS_KWARGS, YEO17_NAMES,
    DMN_INDICES, FPN_INDICES, DAN_INDICES, SALIENCE_INDICES,
    EXCLUDE_SUBJECTS,
)
from figure_style import (
    setup_style, save_fig,
    COLORS, NETWORK_COLORS, FIGURES_DIR,
)
from parcellation import load_atlas

setup_style()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load the block-level features (most figures use this)
block_df = pd.read_csv(BLOCK_FEATURES_CSV)
print(f'block_df: {block_df.shape}')
print(f'  conditions: {block_df["condition"].value_counts().to_dict()}')
print(f'  subjects: {block_df["subject"].nunique()}')
print(f'\nFigures will save to: {FIGURES_DIR}')

## Figure 0: Atlas visualization

Shows the parcellation we used. Useful for methods section.

In [ ]:
from nilearn import plotting

atlas_img, atlas_labels = load_atlas(ATLAS_NAME, ATLAS_KWARGS)
if ATLAS_NAME == 'yeo_2011' and ATLAS_KWARGS.get('n_networks') == 17:
    atlas_labels = list(YEO17_NAMES)

fig = plt.figure(figsize=(7, 2.0))
display = plotting.plot_roi(
    atlas_img,
    title=f'{ATLAS_NAME} ({len(atlas_labels)} parcels)',
    cut_coords=list(range(-30, 60, 15)),
    display_mode='z',
    figure=fig,
    cmap='Set3',
    annotate=True,
    draw_cross=False,
)
save_fig(fig, 'fig00_atlas')
plt.show()

## Figure 1: Trial structure schematic

Stylized timeline of one trial: Thoughtblock (32s) -> SART (~33s) -> Reflect (2s) -> ThoughtProbe (28s) -> Fixation. Use for methods section.

In [ ]:
from matplotlib.patches import FancyBboxPatch

trial_phases = [
    ('Thoughtblock',  0,   32, COLORS['thoughtblock'], 'Hold worry topic in mind'),
    ('SART',         32,   33, COLORS['sart'],         'Sustained attention task'),
    ('Reflect',      65,    2, '#888888',              ''),
    ('ThoughtProbe', 67,   28, '#5BAEC8',              '6 ratings of just-experienced thought'),
    ('Fixation',     95,    8, COLORS['fixation'],     ''),
]

fig, ax = plt.subplots(figsize=(7, 1.8))
for label, onset, duration, color, sublabel in trial_phases:
    ax.add_patch(FancyBboxPatch(
        (onset, 0.2), duration, 0.6,
        boxstyle='round,pad=0,rounding_size=0.5',
        facecolor=color, edgecolor='none', alpha=0.9,
    ))
    ax.text(onset + duration / 2, 0.5, label,
            ha='center', va='center', color='white',
            fontsize=9, fontweight='bold')
    if sublabel:
        ax.text(onset + duration / 2, -0.2, sublabel,
                ha='center', va='top', color='#444444', fontsize=7)
    ax.text(onset + duration / 2, 1.0, f'{duration}s',
            ha='center', va='bottom', fontsize=7, color='#666666')
ax.set_xlim(-2, 105)
ax.set_ylim(-0.7, 1.3)
ax.set_yticks([])
ax.set_xlabel('Time (s)')
ax.spines['left'].set_visible(False)
ax.set_title('Trial structure (1 of 9 trials per run; 3 runs per session)')
plt.tight_layout()
save_fig(fig, 'fig01_trial_structure')
plt.show()

## Figure 2: Per-subject motion + block usability

Two-panel: (a) % spike TRs per subject across runs, (b) % blocks dropped for QC failure per subject. Useful for supplementary / QC reporting.

In [ ]:
atlas_dir = PARCELS_DIR / ATLAS_NAME
qc_rows = []
for prov_path in sorted(atlas_dir.glob('*_provenance.json')):
    with open(prov_path) as f:
        prov = json.load(f)
    qc_rows.append({
        'subject': prov['subject'],
        'run': prov['run'],
        'n_trs': prov['n_trs'],
        'n_spike_trs': prov['n_spike_trs'],
        'pct_spike': 100 * prov['n_spike_trs'] / prov['n_trs'],
    })
qc_df = pd.DataFrame(qc_rows)

per_sub_spike = qc_df.groupby('subject')['pct_spike'].median().sort_values()

MAX_BLOCKS_PER_SUB = 54  # 27 TB + 27 SART (9 thoughts × 3 runs × 2 conditions)
blocks_per_sub = block_df.groupby('subject').size()
blocks_dropped_pct = 100 * (1 - blocks_per_sub / MAX_BLOCKS_PER_SUB)

fig, axes = plt.subplots(1, 2, figsize=(7, 3))

ax = axes[0]
ax.bar(range(len(per_sub_spike)), per_sub_spike.values,
        color=COLORS['real'], width=0.85, edgecolor='none')
ax.set_xlabel('Subject (sorted)')
ax.set_ylabel('% TRs flagged as spikes (median across runs)')
ax.set_title('a. Motion: spike TRs per subject')
ax.axhline(10, color='red', linestyle='--', linewidth=0.5, alpha=0.5,
            label='10% reference')
ax.legend(frameon=False, loc='upper left')

ax = axes[1]
blocks_dropped_pct_sorted = blocks_dropped_pct.sort_values()
ax.bar(range(len(blocks_dropped_pct_sorted)), blocks_dropped_pct_sorted.values,
        color=COLORS['real'], width=0.85, edgecolor='none')
ax.set_xlabel('Subject (sorted)')
ax.set_ylabel('% blocks dropped at QC')
ax.set_title('b. Block usability after spike scrubbing')

plt.tight_layout()
save_fig(fig, 'fig02_qc_summary')
plt.show()

print(f'\nQC summary:')
print(f'  Per-subject median spike %: M={per_sub_spike.mean():.1f}%, '
      f'range {per_sub_spike.min():.1f}-{per_sub_spike.max():.1f}%')
print(f'  Per-subject blocks dropped: M={blocks_dropped_pct.mean():.1f}%, '
      f'range {blocks_dropped_pct.min():.1f}-{blocks_dropped_pct.max():.1f}%')

## Figure 3: Probe-dyscontrol validation

Within-subject correlations between prescan dyscontrol and the 6 in-scanner probe ratings. Validates that probes 1, 4, 5 (which ask about repetition / intrusion) track the dyscontrol construct, and that the merge is correctly aligned.

In [ ]:
tb = block_df[block_df['condition'] == 'Thoughtblock']
probe_cols = [f'probe{i}' for i in range(1, 7)]

results = []
for col in probe_cols:
    if col not in tb.columns:
        continue
    rs = []
    for sub, g in tb.groupby('subject'):
        valid = g.dropna(subset=['dyscontrol_imputed', col])
        if len(valid) < 5 or valid['dyscontrol_imputed'].std() == 0:
            continue
        v = pd.to_numeric(valid[col], errors='coerce')
        if v.std() == 0:
            continue
        r, _ = pearsonr(valid['dyscontrol_imputed'], v)
        if not np.isnan(r):
            rs.append(r)
    rs = np.array(rs)
    results.append({
        'probe': col, 'mean_r': rs.mean(),
        'median_r': np.median(rs), 'sem_r': rs.std() / np.sqrt(len(rs)),
        'n_subj': len(rs), 'pct_pos': (rs > 0).mean() * 100,
        'all_rs': rs,
    })
rdf = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(5, 3))
x = np.arange(len(rdf))
for i, row in rdf.iterrows():
    rng = np.random.default_rng(42 + i)
    jitter = rng.uniform(-0.15, 0.15, size=len(row['all_rs']))
    ax.scatter(x[i] + jitter, row['all_rs'],
                color='#888888', s=6, alpha=0.4, edgecolor='none')
ax.errorbar(x, rdf['mean_r'], yerr=rdf['sem_r'],
             fmt='o', color=COLORS['real'], markersize=7, capsize=3,
             linewidth=1.5, zorder=5)
ax.axhline(0, color='#bbbbbb', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([p.replace('probe', 'Probe ') for p in rdf['probe']])
ax.set_ylabel('Within-subject Pearson r\n(probe × prescan dyscontrol)')
ax.set_title('Probe-dyscontrol within-subject correlations\n'
              '(validates trial-level merge alignment)')
save_fig(fig, 'fig03_probe_dyscontrol_validation')
plt.show()

## Figure 4: LOSO decoding + permutation null

Two-panel: (a) per-subject prediction r distribution as a histogram with mean line, (b) permutation null distribution with real mean overlaid. Headline finding.

In [ ]:
DECODING_DIR = DERIVATIVES_DIR / 'decoding'
loso_per_subj = pd.read_csv(DECODING_DIR / 'loso_per_subject.csv')
perm_null = pd.read_csv(DECODING_DIR / 'loso_permutation_null.csv')

real_rs = loso_per_subj['r'].dropna().values
real_mean = real_rs.mean()
null_means = perm_null['mean_r'].values
p_perm = (np.abs(null_means) >= np.abs(real_mean)).mean()

fig, axes = plt.subplots(1, 2, figsize=(7, 3))

# Panel a: per-subject r distribution
ax = axes[0]
ax.hist(real_rs, bins=15, color=COLORS['real'], alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='#bbbbbb', linewidth=0.5)
ax.axvline(real_mean, color=COLORS['high'], linewidth=1.5, linestyle='--',
            label=f'Mean r = {real_mean:+.3f}')
ax.set_xlabel('Held-out subject prediction r')
ax.set_ylabel('Number of subjects')
ax.set_title(f'a. LOSO ridge (n={len(real_rs)} subjects)')
ax.legend(frameon=False)

# Panel b: permutation null vs real
ax = axes[1]
ax.hist(null_means, bins=30, color=COLORS['permuted'], alpha=0.6,
         edgecolor='white', linewidth=0.5, label='Permutation null')
ax.axvline(real_mean, color=COLORS['high'], linewidth=2, label=f'Real mean = {real_mean:+.3f}')
ax.set_xlabel('Mean r across held-out subjects')
ax.set_ylabel('Number of permutations')
ax.set_title(f'b. Permutation null (n={len(null_means)} perms; p = {p_perm:.3f})')
ax.legend(frameon=False)

plt.tight_layout()
save_fig(fig, 'fig04_loso_decoding')
plt.show()

## Figure 5: Parcel coefficient interpretation

Per-parcel univariate correlation between within-subject z-scored parcel activity and within-subject-centered dyscontrol. Bars colored by the network family (DMN, FPN, DAN, etc.) so the biology is visually grouped.

In [ ]:
coef_df = pd.read_csv(DECODING_DIR / 'parcel_coefficients.csv')
# Sort by univariate r magnitude
coef_df_sorted = coef_df.reindex(coef_df['univariate_r'].abs().sort_values(ascending=True).index)

fig, ax = plt.subplots(figsize=(6, 5))
y_pos = np.arange(len(coef_df_sorted))
colors = [NETWORK_COLORS.get(net, '#888888') for net in coef_df_sorted['network']]
ax.barh(y_pos, coef_df_sorted['univariate_r'], color=colors, edgecolor='none')
ax.axvline(0, color='#444444', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(coef_df_sorted['network'], fontsize=8)
ax.set_xlabel('Univariate Pearson r\n(parcel activity × within-subject dyscontrol)')
ax.set_title('Per-parcel correlations with within-subject dyscontrol')

# Mark significance with asterisks
for i, (_, r) in enumerate(coef_df_sorted.iterrows()):
    if r['univariate_p'] < 0.001:
        sig = '***'
    elif r['univariate_p'] < 0.01:
        sig = '**'
    elif r['univariate_p'] < 0.05:
        sig = '*'
    else:
        sig = ''
    if sig:
        x = r['univariate_r']
        offset = 0.005 if x > 0 else -0.005
        ha = 'left' if x > 0 else 'right'
        ax.text(x + offset, i, sig, va='center', ha=ha, fontsize=7)

plt.tight_layout()
save_fig(fig, 'fig05_parcel_coefficients')
plt.show()

## Figure 6: Decoder × probe correlations

For each retrospective probe (1-6), within-subject correlation between LOSO-decoded dyscontrol and the probe rating. Validates that the decoder picks up the stickiness construct as participants later report it.

In [ ]:
probe_results = pd.read_csv(DECODING_DIR / 'probe_validation.csv')

fig, ax = plt.subplots(figsize=(5, 3))
x = np.arange(len(probe_results))
ax.errorbar(x, probe_results['mean_r'], yerr=probe_results['sem_r'],
             fmt='o', color=COLORS['real'], markersize=7, capsize=3,
             linewidth=1.5)
ax.axhline(0, color='#bbbbbb', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([p.replace('probe', 'Probe ') for p in probe_results['probe']])
ax.set_ylabel('Within-subject Pearson r\n(decoded dyscontrol × probe)')
ax.set_title('Decoder predictions track probe ratings within-subject')

for i, (_, r) in enumerate(probe_results.iterrows()):
    if r['p_value'] < 0.001:
        sig = '***'
    elif r['p_value'] < 0.01:
        sig = '**'
    elif r['p_value'] < 0.05:
        sig = '*'
    else:
        sig = ''
    if sig:
        ax.text(i, r['mean_r'] + r['sem_r'] + 0.005, sig,
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
save_fig(fig, 'fig06_decoder_probe_validation')
plt.show()

In [ ]:
# =============================================================================
# Figure: MVPA workflow explainer (2x2 grid)
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle, Circle, FancyArrow
from matplotlib.lines import Line2D
import nibabel as nib
from nilearn import datasets, plotting

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Color palette
COLORS = {
    'brain':     '#3a4f6e',
    'pattern1':  '#3a7cba',
    'pattern2':  '#7791a8',
    'mean':      '#1f3a5f',
    'highlight': '#C73C3C',
    'arrow':     '#666666',
    'box':       '#dde6ee',
}

# =============================================================================
# Panel 1 (top-left): Brain pattern extraction
# =============================================================================
ax = axes[0, 0]

# Get a small brain slice
yeo = datasets.fetch_atlas_yeo_2011(n_networks=17, thickness='thick')
atlas_img = nib.load(yeo['maps'])
mni_template = datasets.load_mni152_template(resolution=2)

# Use plot_anat as a base, then we'll add overlay
display = plotting.plot_anat(
    mni_template, axes=ax, display_mode='z', cut_coords=[20],
    annotate=False, draw_cross=False, colorbar=False,
)
# Color the atlas on top with a bit of saturation
display.add_overlay(atlas_img, cmap='tab20', alpha=0.5)

ax.set_title('1. Extract activity from brain regions', fontsize=12, loc='left',
              fontweight='bold', pad=10)

# Annotation arrow + caption below
ax.text(0.5, -0.13,
        'For each block, average BOLD signal\nin each of 17 brain networks → '
        '17-dimensional activity pattern',
        transform=ax.transAxes, ha='center', va='top', fontsize=10)

# =============================================================================
# Panel 2 (top-right): Pairing patterns with behavior
# =============================================================================
ax = axes[0, 1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

ax.set_title('2. Pair each pattern with behavior', fontsize=12, loc='left',
              fontweight='bold', pad=10)

# Create 4 example "blocks" with mini parcel patterns + dyscontrol scores
np.random.seed(7)
example_data = [
    {'patt': np.random.randn(17) * 0.5 + 0.3, 'score': 6.3, 'label': 'Block 1'},
    {'patt': np.random.randn(17) * 0.5 - 0.2, 'score': 2.1, 'label': 'Block 2'},
    {'patt': np.random.randn(17) * 0.5 + 0.4, 'score': 5.8, 'label': 'Block 3'},
    {'patt': np.random.randn(17) * 0.5 - 0.5, 'score': 1.2, 'label': 'Block 4'},
]

# Layout: 4 mini-pattern rows
y_top = 8.5
y_step = 1.8
for i, ex in enumerate(example_data):
    y = y_top - i * y_step
    # Block label
    ax.text(0.3, y, ex['label'], fontsize=9, va='center')
    # Mini pattern: 17 colored cells
    cell_w = 0.35
    cell_h = 0.7
    for j, val in enumerate(ex['patt']):
        x = 2.0 + j * cell_w
        # Color by value: blue for negative, red for positive
        c = plt.cm.RdBu_r((val + 1.5) / 3.0)
        rect = Rectangle((x, y - cell_h/2), cell_w, cell_h, facecolor=c,
                          edgecolor='black', linewidth=0.3)
        ax.add_patch(rect)
    # Score on right
    ax.text(8.5, y, f'→ {ex["score"]:.1f}', fontsize=11, va='center',
            fontweight='bold', color=COLORS['mean'])

# Add column labels
ax.text(2.0 + (17 * cell_w) / 2, y_top + 1.0, 'Brain pattern (17 networks)',
        fontsize=10, ha='center', fontweight='normal', style='italic')
ax.text(8.7, y_top + 1.0, 'Dyscontrol\nscore',
        fontsize=10, ha='center', fontweight='normal', style='italic')

# Caption
ax.text(0.5, -0.05,
        'Each block is paired with the dyscontrol\n'
        'score for that thought',
        transform=ax.transAxes, ha='center', va='top', fontsize=10)

# =============================================================================
# Panel 3 (bottom-left): Leave-one-subject-out cross-validation
# =============================================================================
ax = axes[1, 0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

ax.set_title('3. Leave-one-subject-out cross-validation', fontsize=12,
              loc='left', fontweight='bold', pad=10)

# Draw 8 stick figures — 7 in training (blue), 1 held out (red)
n_subj = 8
for i in range(n_subj):
    cx = 1.0 + i * 1.05
    cy = 7.5
    
    # Held out subject = subject 4 (index 3)
    is_held_out = (i == 3)
    color = COLORS['highlight'] if is_held_out else COLORS['pattern1']
    label = 'Test' if is_held_out else 'Train'
    
    # Head
    head = Circle((cx, cy), 0.22, facecolor=color, edgecolor='black', linewidth=0.5)
    ax.add_patch(head)
    # Body
    ax.plot([cx, cx], [cy - 0.22, cy - 0.95], color=color, linewidth=2)
    # Arms
    ax.plot([cx - 0.4, cx + 0.4], [cy - 0.55, cy - 0.55], color=color, linewidth=2)
    # Legs
    ax.plot([cx, cx - 0.3], [cy - 0.95, cy - 1.5], color=color, linewidth=2)
    ax.plot([cx, cx + 0.3], [cy - 0.95, cy - 1.5], color=color, linewidth=2)

# Brackets to indicate train + test
ax.annotate('', xy=(0.6, 5.5), xytext=(3.4, 5.5),
            arrowprops=dict(arrowstyle='-', color='black', lw=1))
ax.annotate('', xy=(4.05, 5.5), xytext=(4.05, 5.5),
            arrowprops=dict(arrowstyle='-', color='black', lw=1))
ax.annotate('', xy=(4.45, 5.5), xytext=(8.4, 5.5),
            arrowprops=dict(arrowstyle='-', color='black', lw=1))
ax.text(2.0, 5.0, 'Train decoder\non 56 subjects',
        ha='center', fontsize=10, color=COLORS['pattern1'])
ax.text(4.05, 5.0, '...\nplus sub 4',
        ha='center', fontsize=10, color='gray', alpha=0)
ax.text(6.5, 5.0, 'Test on the\nheld-out subject',
        ha='center', fontsize=10, color=COLORS['highlight'])

# Highlight subject 4 with arrow
ax.annotate('', xy=(4.15, 6.7), xytext=(4.15, 4.0),
            arrowprops=dict(arrowstyle='-|>', color=COLORS['highlight'], lw=2))
ax.text(4.15, 8.7, 'Held out',
        ha='center', fontsize=9, color=COLORS['highlight'], fontweight='bold')

# Repeat indicator
ax.text(5, 2.8,
        'Repeat 57 times — each subject held out once',
        ha='center', fontsize=11, fontweight='bold')

# Caption
ax.text(0.5, -0.05,
        'The model never sees test data during training.\n'
        'This guards against overfitting and inflated accuracy.',
        transform=ax.transAxes, ha='center', va='top', fontsize=10)

# =============================================================================
# Panel 4 (bottom-right): Group-level test
# =============================================================================
ax = axes[1, 1]

ax.set_title('4. Test if predictions are better than chance', fontsize=12,
              loc='left', fontweight='bold', pad=10)

# Generate fake-but-realistic per-subject r values
# Mean ~0.099 like real TB decoder, SD 0.16, n=57
np.random.seed(42)
fake_rs = np.random.normal(loc=0.099, scale=0.16, size=57)

# Strip plot of per-subject correlations
np.random.seed(42)
jitter = np.random.uniform(-0.18, 0.18, size=len(fake_rs))
ax.scatter(jitter, fake_rs, s=28, color=COLORS['pattern1'], alpha=0.6,
            edgecolor='white', linewidth=0.5, zorder=2)

mean_r = fake_rs.mean()
se_r = fake_rs.std() / np.sqrt(len(fake_rs))
ax.hlines(mean_r, -0.30, 0.30, color=COLORS['mean'], linewidth=2.5, zorder=3)
ax.vlines(0, mean_r - se_r, mean_r + se_r, color=COLORS['mean'], linewidth=2,
            zorder=3)

# Reference line at zero
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', zorder=1)

# Annotation
ax.text(0.45, mean_r + 0.01, 'Group mean\n(better than chance)',
        ha='left', va='center', fontsize=10,
        color=COLORS['mean'], fontweight='bold')

ax.set_xticks([0])
ax.set_xticklabels(['57 subjects\n(one dot each)'], fontsize=10)
ax.set_xlim(-0.6, 1.5)
ax.set_ylabel('Pearson r between\npredicted and actual scores', fontsize=10)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Caption
ax.text(0.5, -0.13,
        'If the mean correlation across subjects is\n'
        'reliably above zero, the brain pattern carries\n'
        'information about behavior.',
        transform=ax.transAxes, ha='center', va='top', fontsize=10)

# =============================================================================
# Overall title and save
# =============================================================================
plt.suptitle('Multivariate Pattern Analysis (MVPA): how it works',
              fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout()
plt.subplots_adjust(top=0.93, hspace=0.55, wspace=0.25)

fig_path = FIGURES_DIR / 'fig_mvpa_workflow_explainer.png'
#plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')